# This notebook demonstrates the core idea of Praxis-BGM:

Learn cluster structure on a SOURCE dataset → treat the learned posteriors as PRIORS → transfer them to a TARGET dataset → achieve better clustering.


##  Install and basic imports

In [2]:
!pip install git+https://github.com/ContiLab-usc/Praxis-BGM.git

  Cloning https://github.com/ContiLab-usc/Praxis-BGM.git to /tmp/pip-req-build-ocy0fmgh
  Running command git clone --filter=blob:none --quiet https://github.com/ContiLab-usc/Praxis-BGM.git /tmp/pip-req-build-ocy0fmgh
  Resolved https://github.com/ContiLab-usc/Praxis-BGM.git to commit da2cbce634872fc331381942bdec6306a1aac779
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for praxis-bgm: filename=praxis_bgm-0.2.0-py3-none-any.whl size=40038 sha256=efa57a3e0c1f4118a185599ce806c49cef98db771582f3937951f8f225133a7a
  Stored in directory: /tmp/pip-ephem-wheel-cache-cagx7tjo/wheels/17/c4/6d/2be8e0203711040b22445ec499ab0abf56c3cd1e05ef237b75
Successfully built praxis-bgm


In [5]:
import numpy as np
import jax
import jax.numpy as jnp
from jax.random import PRNGKey, split
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Import Praxis_BGM (adjust this import path to match your package structure)
from praxis_bgm import Praxis_BGM

##  Simulate SOURCE data

In [6]:
import numpy as np
from sklearn.metrics import adjusted_rand_score
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

from jax.random import PRNGKey, split
from praxis_bgm import Praxis_BGM


def generate_overlapping_gmm_samples(
    n_components, n_causal, n_features, n_samples, mean_shift=0.4, random_seed=None
):
    """
    SOURCE/TARGET generator:
    - First n_causal features are 'causal' and get cluster-specific means
    - Remaining features are pure noise N(0,1)
    - Covariance is identity, but clusters overlap strongly via small mean_shift
    """
    if random_seed is not None:
        np.random.seed(random_seed)

    base_mean = np.zeros(n_causal)

    # Cluster-specific means only on causal features
    causal_means = np.array([
        base_mean + np.random.randn(n_causal) * mean_shift
        for _ in range(n_components)
    ])

    samples_list, labels_list = [], []
    samples_per_component = n_samples // n_components

    for k in range(n_components):
        causal = np.random.multivariate_normal(
            causal_means[k], np.eye(n_causal), samples_per_component
        )
        non_causal = np.random.normal(
            0, 1, size=(samples_per_component, n_features - n_causal)
        )
        X = np.hstack([causal, non_causal])
        samples_list.append(X)
        labels_list.extend([k] * samples_per_component)

    samples = np.vstack(samples_list)
    labels = np.array(labels_list)

    full_means = np.zeros((n_components, n_features))
    full_means[:, :n_causal] = causal_means

    return samples, labels, full_means, None


def randomly_shift_means(true_means, shift_magnitude, percentage, random_seed):
    """
    Take true means (K x P), randomly choose a percentage of features,
    and add a Gaussian shift of size 'shift_magnitude' on those features.
    This simulates domain shift in feature space.
    """
    np.random.seed(random_seed)
    shifted = np.copy(true_means)
    n_clusters, n_features = shifted.shape
    n_shift = int(percentage * n_features)
    for k in range(n_clusters):
        idx = np.random.choice(n_features, size=n_shift, replace=False)
        shifted[k, idx] += np.random.randn(n_shift) * shift_magnitude
    return shifted


def l2_norm_with_alignment(est_means, true_means):
    """
    Align clusters using Hungarian algorithm and compute L2 distance
    between estimated and true means.
    """
    cost = cdist(est_means, true_means)
    row_ind, col_ind = linear_sum_assignment(cost)
    aligned_est = est_means[row_ind]
    aligned_true = true_means[col_ind]
    return np.linalg.norm(aligned_est - aligned_true)

In [7]:
# SOURCE settings (overlapping, moderately high-dim)
K = 4          # clusters
P = 100        # total features
C = 40         # causal features
N_src = 800    # larger source sample size


X_src, y_src, true_means_src, _ = generate_overlapping_gmm_samples(
    n_components=K,
    n_causal=C,
    n_features=P,
    n_samples=N_src,
    random_seed=123
)

print("X_src shape:", X_src.shape)
print("True means (source) shape:", true_means_src.shape)

X_src shape: (800, 100)
True means (source) shape: (4, 100)


## Learning Priors from SOURCE

In [8]:
print("\n=== Compute empirical SOURCE means/covs from labels (for priors) ===")

mus_post_src = []
covs_post_src = []

eps = 1e-4  # small jitter to make covariances numerically stable

for k in range(K):
    Xk = X_src[y_src == k]
    if Xk.shape[0] == 0:
        raise ValueError(f"No samples for cluster {k} in source data!")

    mu_k = Xk.mean(axis=0)


    mus_post_src.append(mu_k)


mus_post_src = np.stack(mus_post_src).astype(np.float32)   # shape (K, P)


# Also get empirical mixture weights from labels
pis_src_emp = np.bincount(y_src, minlength=K) / len(y_src)

print("Empirical SOURCE π:", np.round(pis_src_emp, 3))
print("Empirical SOURCE means shape:", mus_post_src.shape)



=== Compute empirical SOURCE means/covs from labels (for priors) ===
Empirical SOURCE π: [0.25 0.25 0.25 0.25]
Empirical SOURCE means shape: (4, 100)


## run a QDA baseline trained on source and predict on targe

In [9]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
# --- QDA baseline: train on SOURCE labels ---
qda = QuadraticDiscriminantAnalysis(store_covariance=True)
qda.fit(X_src, y_src)
print("QDA fitted on SOURCE.")

QDA fitted on SOURCE.


## Simulate TARGET data (domain shift)

In [10]:
# TARGET settings: smaller N, stronger mean shift, many shifted features
N_tgt = 200
mean_shift_tgt = 0.3      # stronger magnitude than source
shift_percentage = 0.3    # 70% of features shifted

# Build target true means as a shifted version of the *source* true means
true_means_tgt = randomly_shift_means(
    true_means_src,
    shift_magnitude=mean_shift_tgt,
    percentage=shift_percentage,
    random_seed=999
)

# Generate TARGET data using those shifted means
np.random.seed(999)
samples_per_component_tgt = N_tgt // K

X_tgt_list, y_tgt_list = [], []
for k in range(K):
    # Causal part: multivariate normal around shifted means
    causal_tgt = np.random.multivariate_normal(
        true_means_tgt[k, :C],
        np.eye(C),
        samples_per_component_tgt
    )
    # Non-causal part: pure noise
    non_causal_tgt = np.random.normal(
        0, 1, size=(samples_per_component_tgt, P - C)
    )
    Xk = np.hstack([causal_tgt, non_causal_tgt])
    X_tgt_list.append(Xk)
    y_tgt_list.extend([k] * samples_per_component_tgt)

X_tgt = np.vstack(X_tgt_list)
y_tgt = np.array(y_tgt_list)

print("X_tgt shape:", X_tgt.shape)
print("True means (target) shape:", true_means_tgt.shape)


X_tgt shape: (200, 100)
True means (target) shape: (4, 100)


In [11]:
# --- QDA baseline: predict TARGET using supervised source model ---
print("\n=== QDA baseline: SOURCE-supervised model on TARGET ===")
yhat_qda = qda.predict(X_tgt)
ari_qda = adjusted_rand_score(y_tgt, yhat_qda)
print(f"[TARGET] ARI QDA (source-trained, supervised): {ari_qda:.3f}")


=== QDA baseline: SOURCE-supervised model on TARGET ===
[TARGET] ARI QDA (source-trained, supervised): 0.101


## TARGET: Praxis-BGM with transferred priors

In [16]:
print("\n=== Fit Praxis-BGM on TARGET with TRANSFERRED PRIORS ===")




model_t_prior = Praxis_BGM(
    rng_key=jax.random.PRNGKey(0),
    K=K,
    prior_mus=mus_post_src,                # informative mean priors from SOURCE
    beta=1e-4,
    tol=1e-4,
    max_iters=300,
    verbose=True,
    prior_mus_variance=1.,                # stronger trust in prior means
    num_samples=64,
)

model_t_prior.fit(X_tgt, num_iters=120, batch_size=64, early_stop=True, patience=5)
yhat_prior, _ = model_t_prior.predict(X_tgt)
ari_prior = adjusted_rand_score(y_tgt, yhat_prior)
print(f"[TARGET] ARI with transferred priors: {ari_prior:.3f}")


=== Fit Praxis-BGM on TARGET with TRANSFERRED PRIORS ===
[FIT] Starting training with data shape = (200, 100), K = 4
  => Checking user priors & setting up initial state (num_samples = 64)...
[Init Priors] Checking user-provided priors...
  => Only prior means provided.
[Init Params] Initializing mixture parameters...
  => Attempt #1 with current priors.
[FIT] Iter 10/120 | MC ELBO ≈ -34999.824219 | Δ ≈ inf | data term ≈ -34993.796875 (-174.968979/obs) | reg term ≈ 6.027860 (1.506965/comp) | avg h(z) ≈ 553.826904 | beta ≈ 1.000e-04
[FIT] Iter 20/120 | MC ELBO ≈ -34523.960938 | Δ ≈ 475.863281 | data term ≈ -34515.625000 (-172.578125/obs) | reg term ≈ 8.335182 (2.083796/comp) | avg h(z) ≈ 552.339966 | beta ≈ 1.000e-04
[FIT] Iter 30/120 | MC ELBO ≈ -34277.578125 | Δ ≈ 246.382812 | data term ≈ -34268.027344 (-171.340134/obs) | reg term ≈ 9.551048 (2.387762/comp) | avg h(z) ≈ 552.378906 | beta ≈ 1.000e-04
[FIT] Iter 40/120 | MC ELBO ≈ -34284.273438 | Δ ≈ -6.695312 | data term ≈ -34274.9140

## Fit TARGET without priors (baseline)

In [18]:
print("\n=== Fit Praxis-BGM on TARGET without priors (baseline) ===")

model_t_noprior = Praxis_BGM(
    rng_key=jax.random.PRNGKey(0),
    K=K,
    prior_mus=None,
    prior_Sigmas=None,
    beta=1e-3,
    tol=1e-4,
    max_iters=300,
    verbose=True,
    prior_mus_variance=10.0,
    num_samples=64,
)

model_t_noprior.fit(X_tgt, num_iters=120, batch_size=64, early_stop=True, patience=5)
yhat_noprior, _ = model_t_noprior.predict(X_tgt)
ari_noprior = adjusted_rand_score(y_tgt, yhat_noprior)
print(f"[TARGET] ARI without priors: {ari_noprior:.3f}")


=== Fit Praxis-BGM on TARGET without priors (baseline) ===
[FIT] Starting training with data shape = (200, 100), K = 4
  => Checking user priors & setting up initial state (num_samples = 64)...
[Init Priors] Checking user-provided priors...
  => No user priors, using BGM(full) means + identity cov.
[Init Params] Initializing mixture parameters...
  => Attempt #1 with current priors.
[FIT] Iter 10/120 | MC ELBO ≈ -35635.500000 | Δ ≈ inf | data term ≈ -35629.578125 (-178.147888/obs) | reg term ≈ 5.922077 (1.480519/comp) | avg h(z) ≈ 560.736328 | beta ≈ 1.000e-03
[FIT] Iter 20/120 | MC ELBO ≈ -34952.851562 | Δ ≈ 682.648438 | data term ≈ -34944.722656 (-174.723618/obs) | reg term ≈ 8.127805 (2.031951/comp) | avg h(z) ≈ 558.496704 | beta ≈ 1.000e-03
  [EarlyStop] No change in assignments: 1/5
[FIT] Iter 30/120 | MC ELBO ≈ -34622.660156 | Δ ≈ 330.191406 | data term ≈ -34613.359375 (-173.066803/obs) | reg term ≈ 9.301441 (2.325360/comp) | avg h(z) ≈ 557.003418 | beta ≈ 1.000e-03
[FIT] Iter 4

## Final comparison

In [19]:

print("\n=== SUMMARY ===")
print(f"ARI NGVI (with transferred priors): {ari_prior:.3f}")
print(f"ARI NGVI (no priors):             {ari_noprior:.3f}")
print(f"[TARGET] ARI QDA (source-trained, supervised): {ari_qda:.3f}")


=== SUMMARY ===
ARI NGVI (with transferred priors): 0.812
ARI NGVI (no priors):             0.610
[TARGET] ARI QDA (source-trained, supervised): 0.101
